# Superstore Sales Analysis (Pandas + NumPy)

Analysis and data preparation in Python. The dashboard is built in Power BI using the cleaned file exported at the end of this notebook.

**Main questions**
1. Which city has the highest sales?
2. Which category has the highest sales?
3. Who are the top 10 customers?
4. Additional: sales trend, market/region, segment, sub-category, discount vs profit

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Load the data

In [2]:
df = pd.read_csv("../data/global_superstore.csv", encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,42433,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,TB-11280,Toby Braunhardt,Consumer,Constantine,Constantine,...,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408.300,2,0.0,106.140,35.46,Medium
1,22253,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,JH-15985,Joseph Holt,Consumer,Wagga Wagga,New South Wales,...,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.366,3,0.1,36.036,9.72,Medium
2,48883,HU-2011-1220,1/1/2011,5/1/2011,Second Class,AT-735,Annie Thurman,Consumer,Budapest,Budapest,...,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66.120,4,0.0,29.640,8.17,High
3,11731,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,EM-14140,Eugene Moren,Home Office,Stockholm,Stockholm,...,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",44.865,3,0.5,-26.055,4.82,High
4,22255,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,JH-15985,Joseph Holt,Consumer,Wagga Wagga,New South Wales,...,FUR-FU-10003447,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",113.670,5,0.1,37.770,4.70,Medium


In [3]:
print(df.shape)
df.info()

(51290, 24)
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  str    
 2   Order Date      51290 non-null  str    
 3   Ship Date       51290 non-null  str    
 4   Ship Mode       51290 non-null  str    
 5   Customer ID     51290 non-null  str    
 6   Customer Name   51290 non-null  str    
 7   Segment         51290 non-null  str    
 8   City            51290 non-null  str    
 9   State           51290 non-null  str    
 10  Country         51290 non-null  str    
 11  Postal Code     9994 non-null   float64
 12  Market          51290 non-null  str    
 13  Region          51290 non-null  str    
 14  Product ID      51290 non-null  str    
 15  Category        51290 non-null  str    
 16  Sub-Category    51290 non-null  str    
 17  Product Name    51290 non-null

In [4]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit,Shipping Cost
count,51290.00000,9994.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000
mean,25645.50000,55190.379428,246.490581,3.476545,0.142908,28.610982,26.375915
std,14806.29199,32063.693350,487.565361,2.278766,0.212280,174.340972,57.296804
min,1.00000,1040.000000,0.444000,1.000000,0.000000,-6599.978000,0.000000
25%,12823.25000,23223.000000,30.758625,2.000000,0.000000,0.000000,2.610000
50%,25645.50000,56430.500000,85.053000,3.000000,0.000000,9.240000,7.790000
75%,38467.75000,90008.000000,251.053200,5.000000,0.200000,36.810000,24.450000
max,51290.00000,99301.000000,22638.480000,14.000000,0.850000,8399.976000,933.570000


## 3. Data cleaning

In [5]:
# Missing values and duplicates
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
Postal Code    41296
dtype: int64

Duplicate rows: 0


In [6]:
# Postal Code is missing for most rows and is not needed -> drop it
df = df.drop(columns=["Postal Code", "Row ID"])

# Make column names easy to use: "Order Date" -> "Order_Date", "Sub-Category" -> "Sub_Category"
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")
df.columns.tolist()

['Order_ID',
 'Order_Date',
 'Ship_Date',
 'Ship_Mode',
 'Customer_ID',
 'Customer_Name',
 'Segment',
 'City',
 'State',
 'Country',
 'Market',
 'Region',
 'Product_ID',
 'Category',
 'Sub_Category',
 'Product_Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit',
 'Shipping_Cost',
 'Order_Priority']

In [7]:
# Dates come in two styles ("1/1/2011" and "13-01-2011"), so make them uniform first
for col in ["Order_Date", "Ship_Date"]:
    df[col] = pd.to_datetime(df[col].str.replace("-", "/"), format="%d/%m/%Y")

print(df["Order_Date"].min(), "to", df["Order_Date"].max())

2011-01-01 00:00:00 to 2014-12-31 00:00:00


## 4. Feature engineering (new columns)

In [8]:
# Date features
df["Year"] = df["Order_Date"].dt.year
df["Month_No"] = df["Order_Date"].dt.month
df["Month"] = df["Order_Date"].dt.month_name()

# Delivery time in days
df["Delivery_Days"] = (df["Ship_Date"] - df["Order_Date"]).dt.days

# Profit margin %
df["Profit_Margin_%"] = np.round(df["Profit"] / df["Sales"] * 100, 2)

# Loss-making flag (NumPy where)
df["Is_Loss"] = np.where(df["Profit"] < 0, "Loss", "Profit")

# Discount bands (NumPy select)
conditions = [df["Discount"] == 0, df["Discount"] <= 0.2, df["Discount"] <= 0.4]
labels = ["No Discount", "Low (1-20%)", "Medium (21-40%)"]
df["Discount_Band"] = np.select(conditions, labels, default="High (>40%)")

# City with country, because the same city name can exist in different countries
df["City_Country"] = df["City"] + ", " + df["Country"]

df[["Order_Date", "Delivery_Days", "Profit_Margin_%", "Is_Loss", "Discount_Band"]].head()

,Order_Date,Delivery_Days,Profit_Margin_%,Is_Loss,Discount_Band
0,2011-01-01,5,26.00,Profit,No Discount
1,2011-01-01,7,29.94,Profit,Low (1-20%)
2,2011-01-01,4,44.83,Profit,No Discount
3,2011-01-01,4,-58.07,Loss,High (>40%)
4,2011-01-01,7,33.23,Profit,Low (1-20%)


In [9]:
# Outlier orders using the IQR method (NumPy)
q1, q3 = np.percentile(df["Sales"], [25, 75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
df["Sales_Outlier"] = np.where(df["Sales"] > upper, "Yes", "No")

print(f"Upper limit for normal orders: {upper:,.0f}")
print(df["Sales_Outlier"].value_counts())

Upper limit for normal orders: 581
Sales_Outlier
No     45635
Yes     5655
Name: count, dtype: int64


## 5. Key business numbers (KPIs)

In [10]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order_ID"].nunique()
total_customers = df["Customer_ID"].nunique()

print(f"Total Sales      : {total_sales:,.0f}")
print(f"Total Profit     : {total_profit:,.0f}")
print(f"Profit Margin    : {total_profit / total_sales * 100:.1f}%")
print(f"Total Orders     : {total_orders:,}")
print(f"Total Customers  : {total_customers:,}")
print(f"Avg Order Value  : {total_sales / total_orders:,.0f}")

Total Sales      : 12,642,502
Total Profit     : 1,467,457
Profit Margin    : 11.6%
Total Orders     : 25,035
Total Customers  : 1,590
Avg Order Value  : 505


## 6. Analysis

### Q1. Which city has the highest sales?

In [11]:
city_sales = (df.groupby("City_Country")
                .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "nunique"))
                .sort_values("Sales", ascending=False)
                .round(0))

top_city = city_sales.index[0]
print("Top city by sales:", top_city)
city_sales.head(10)

Top city by sales: New York City, United States


,Sales,Profit,Orders
City_Country,,,
"New York City, United States",256368.0,62037.0,450
"Los Angeles, United States",175851.0,30441.0,384
"Manila, Philippines",120887.0,-11159.0,206
"Seattle, United States",119541.0,29156.0,212
"San Francisco, United States",112669.0,17507.0,265
"Philadelphia, United States",109077.0,-13838.0,265
"Sydney, Australia",101946.0,16003.0,131
"Jakarta, Indonesia",94321.0,3827.0,164
"Mexico City, Mexico",85729.0,13342.0,162


### Q2. Which category has the highest sales?

In [12]:
cat_sales = (df.groupby("Category")
               .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Quantity=("Quantity", "sum"))
               .sort_values("Sales", ascending=False))
cat_sales["Sales_Share_%"] = (cat_sales["Sales"] / cat_sales["Sales"].sum() * 100).round(1)
cat_sales["Profit_Margin_%"] = (cat_sales["Profit"] / cat_sales["Sales"] * 100).round(1)

print("Top category by sales:", cat_sales.index[0])
cat_sales.round(1)

Top category by sales: Technology


,Sales,Profit,Quantity,Sales_Share_%,Profit_Margin_%
Category,,,,,
Technology,4744557.5,663778.7,35176,37.5,14.0
Furniture,4110874.2,285204.7,34954,32.5,6.9
Office Supplies,3787070.2,518473.8,108182,30.0,13.7


In [13]:
# Top 10 sub-categories
sub_sales = (df.groupby(["Category", "Sub_Category"])
               .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
               .sort_values("Sales", ascending=False)
               .round(0))
sub_sales.head(10)

Sales    Profit
Category        Sub_Category                     
Technology      Phones        1706824.0  216717.0
                Copiers       1509436.0  258568.0
Furniture       Chairs        1501682.0  140396.0
                Bookcases     1466572.0  161924.0
Office Supplies Storage       1127086.0  108461.0
                Appliances    1011064.0  141681.0
Technology      Machines       779060.0   58868.0
Furniture       Tables         757042.0  -64083.0
Technology      Accessories    749237.0  129626.0
Office Supplies Binders        461912.0   72450.0

### Q3. Who are the top 10 customers?

In [14]:
top_customers = (df.groupby(["Customer_ID", "Customer_Name"])
                   .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "nunique"))
                   .sort_values("Sales", ascending=False)
                   .head(10)
                   .round(0)
                   .reset_index())
top_customers.index = top_customers.index + 1   # rank 1 to 10
top_customers

,Customer_ID,Customer_Name,Sales,Profit,Orders
1,TA-21385,Tom Ashbrook,35668.0,6275.0,25
2,GT-14710,Greg Tran,34472.0,5165.0,30
3,TC-20980,Tamara Chand,34218.0,8787.0,28
4,SM-20320,Sean Miller,31125.0,-1084.0,21
5,BW-11110,Bart Watters,30614.0,3337.0,35
6,HL-15040,Hunter Lopez,29664.0,7658.0,20
7,SE-20110,Sanjit Engle,29533.0,5864.0,36
8,PS-19045,Penelope Sewall,29252.0,4426.0,26
9,RB-19360,Raymond Buch,29198.0,8524.0,25
10,ZC-21910,Zuschuss Carroll,28473.0,453.0,37


### Q4. How do sales change over time?

In [15]:
yearly = df.groupby("Year").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
yearly["Sales_Growth_%"] = (yearly["Sales"].pct_change() * 100).round(1)
yearly.round(1)

,Sales,Profit,Sales_Growth_%
Year,,,
2011,2259450.9,248940.8,NaN
2012,2677438.7,307415.3,18.5
2013,3405746.4,406935.2,27.2
2014,4299865.9,504166.0,26.3


In [16]:
monthly = (df.groupby(["Month_No", "Month"])["Sales"].sum()
             .reset_index().sort_values("Month_No"))
monthly["Share_%"] = (monthly["Sales"] / monthly["Sales"].sum() * 100).round(1)
monthly.round(0)

,Month_No,Month,Sales,Share_%
0,1,January,675134.0,5.0
1,2,February,543739.0,4.0
2,3,March,770501.0,6.0
3,4,April,698561.0,6.0
4,5,May,904012.0,7.0
5,6,June,1269717.0,10.0
6,7,July,749382.0,6.0
7,8,August,1293833.0,10.0
8,9,September,1437380.0,11.0
9,10,October,1168184.0,9.0


### Q5. Which market and region perform best?

In [17]:
market = (df.groupby("Market")
            .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
            .sort_values("Sales", ascending=False))
market["Profit_Margin_%"] = (market["Profit"] / market["Sales"] * 100).round(1)
market.round(1)

,Sales,Profit,Profit_Margin_%
Market,,,
APAC,3585744.1,436000.0,12.2
EU,2938089.1,372829.7,12.7
US,2297200.9,286397.0,12.5
LATAM,2164605.2,221643.5,10.2
EMEA,806161.3,43898.0,5.4
Africa,783773.2,88871.6,11.3
Canada,66928.2,17817.4,26.6


In [18]:
region = (df.groupby("Region")
            .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
            .sort_values("Sales", ascending=False)
            .round(0))
region

,Sales,Profit
Region,,
Central,2822303.0,311404.0
South,1600907.0,140356.0
North,1248166.0,194598.0
Oceania,1100185.0,120089.0
Southeast Asia,884423.0,17852.0
North Asia,848310.0,165578.0
EMEA,806161.0,43898.0
Africa,783773.0,88872.0
Central Asia,752827.0,132480.0


### Q6. Which customer segment and shipping mode matter most?

In [19]:
segment = (df.groupby("Segment")
             .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Customers=("Customer_ID", "nunique"))
             .sort_values("Sales", ascending=False)
             .round(0))
segment

,Sales,Profit,Customers
Segment,,,
Consumer,6507949.0,749240.0,818
Corporate,3824698.0,441208.0,476
Home Office,2309855.0,277009.0,296


In [20]:
ship = (df.groupby("Ship_Mode")
          .agg(Orders=("Order_ID", "nunique"), Avg_Delivery_Days=("Delivery_Days", "mean"))
          .round(1))
ship

,Orders,Avg_Delivery_Days
Ship_Mode,,
First Class,3821,2.2
Same Day,1347,0.0
Second Class,5119,3.2
Standard Class,15154,5.0


### Q7. Does discount hurt profit?

In [21]:
discount = (df.groupby("Discount_Band")
              .agg(Orders=("Order_ID", "count"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
              .reindex(["No Discount", "Low (1-20%)", "Medium (21-40%)", "High (>40%)"]))
discount["Profit_Margin_%"] = (discount["Profit"] / discount["Sales"] * 100).round(1)
discount.round(1)

,Orders,Sales,Profit,Profit_Margin_%
Discount_Band,,,,
No Discount,29009,6992411.0,1770695.3,25.3
Low (1-20%),10953,3719880.2,511444.1,13.7
Medium (21-40%),4367,1083897.8,-187271.1,-17.3
High (>40%),6961,846313.0,-627411.0,-74.1


In [22]:
# Share of loss-making orders and correlation
print("Loss-making rows: {:.1f}%".format((df["Is_Loss"] == "Loss").mean() * 100))
df[["Sales", "Quantity", "Discount", "Profit", "Shipping_Cost"]].corr().round(2)

Loss-making rows: 24.5%


,Sales,Quantity,Discount,Profit,Shipping_Cost
Sales,1.00,0.31,-0.09,0.48,0.77
Quantity,0.31,1.00,-0.02,0.10,0.27
Discount,-0.09,-0.02,1.00,-0.32,-0.08
Profit,0.48,0.10,-0.32,1.00,0.35
Shipping_Cost,0.77,0.27,-0.08,0.35,1.00


## 7. Key insights (generated from the data)

In [23]:
print(f"1. Highest sales city      : {city_sales.index[0]} ({city_sales['Sales'].iloc[0]:,.0f})")
print(f"2. Highest sales category   : {cat_sales.index[0]} ({cat_sales['Sales_Share_%'].iloc[0]}% of total sales)")
print(f"3. Top customer             : {top_customers['Customer_Name'].iloc[0]} ({top_customers['Sales'].iloc[0]:,.0f})")
print(f"4. Top 10 customers share   : {top_customers['Sales'].sum() / total_sales * 100:.1f}% of total sales")
print(f"5. Best year                : {yearly['Sales'].idxmax()} (growth {yearly['Sales_Growth_%'].iloc[-1]}% over previous year)")
print(f"6. Best market              : {market.index[0]}")
print(f"7. Most profitable category : {cat_sales['Profit'].idxmax()}")
print(f"8. Discount effect          : profit margin is {discount.loc['No Discount', 'Profit_Margin_%']}% with no discount "
      f"vs {discount.loc['High (>40%)', 'Profit_Margin_%']}% with a high discount")

1. Highest sales city      : New York City, United States (256,368)
2. Highest sales category   : Technology (37.5% of total sales)
3. Top customer             : Tom Ashbrook (35,668)
4. Top 10 customers share   : 2.5% of total sales
5. Best year                : 2014 (growth 26.3% over previous year)
6. Best market              : APAC
7. Most profitable category : Technology
8. Discount effect          : profit margin is 25.3% with no discount vs -74.1% with a high discount


## 8. Export cleaned data for Power BI

In [24]:
df.to_csv("../data/superstore_cleaned.csv", index=False)
print("Saved: data/superstore_cleaned.csv", df.shape)

Saved: data/superstore_cleaned.csv (51290, 31)
